# VietEmbed-RAG V1 — VN-MTEB RAG Core Benchmark

Notebook dành riêng cho **VietEmbed-RAG V1**, model fine-tune cho retrieval/RAG.

Mục tiêu:
- Chỉ benchmark Retrieval
- Không chạy các corpus khổng lồ như MSMARCO, HotpotQA, FEVER, NQ
- Chạy thực tế trên Google Colab
- Dễ debug
- Chạy từng task một
- Có cache để chạy tiếp nếu Colab bị ngắt
- Dùng FP16 trên GPU

## Bộ benchmark mặc định

| Task | Corpus | Ý nghĩa |
|---|---:|---|
| SciFact-VN | ~5K | Scientific retrieval |
| NFCorpus-VN | ~10K | Medical retrieval |
| CQADupstackAndroid-VN | ~25K | Technical / Web QA |
| SCIDOCS-VN | ~38K | Academic retrieval |
| FiQA2018-VN | ~59K | Finance QA |
| TRECCOVID-VN | ~229K | Medical / Academic retrieval |
| Quora-VN | ~534K | General Web QA |

6 task lõi: khoảng 365K documents.  
Bật Quora: khoảng 900K documents.

## 1. Cài thư viện

In [ ]:
!pip install -q -U "mteb[xet]>=2.2.0" sentence-transformers

## 2. Kiểm tra môi trường

In [ ]:
import torch
import mteb
import sentence_transformers

print("PyTorch:", torch.__version__)
print("MTEB:", mteb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cấu hình

Thường chỉ cần sửa `MODEL_ZIP`.

- `INCLUDE_QUORA = True`: benchmark đầy đủ hơn cho general RAG
- `INCLUDE_QUORA = False`: test nhanh

In [ ]:
MODEL_ZIP = "/content/VietEmbed-RAG-V1.zip"
EXTRACT_DIR = "/content/VietEmbed-RAG-V1"

USE_E5_PREFIX = True
BATCH_SIZE = 32

INCLUDE_QUORA = True

RESULT_DIR = "/content/vietembed_v1_mteb_results"

## 4. Giải nén model

In [ ]:
from pathlib import Path
import shutil

shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
shutil.unpack_archive(MODEL_ZIP, EXTRACT_DIR)

model_files = list(Path(EXTRACT_DIR).rglob("modules.json"))

if not model_files:
    raise FileNotFoundError(
        "Không tìm thấy modules.json. Kiểm tra lại file model .zip."
    )

MODEL_DIR = str(model_files[0].parent)

print("Model directory:", MODEL_DIR)

## 5. Load model

Với E5 retrieval:
- Query: `query: ...`
- Document: `passage: ...`

Nếu có GPU, model chuyển sang FP16.

In [ ]:
from sentence_transformers import SentenceTransformer

prompts = {}

if USE_E5_PREFIX:
    prompts = {
        "query": "query: ",
        "document": "passage: ",
    }

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    MODEL_DIR,
    device=device,
    prompts=prompts,
)

if device == "cuda":
    model.half()

model.eval()

print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Max sequence length:", model.max_seq_length)
print("Prompts:", prompts)

## 6. Sanity check

In [ ]:
from sentence_transformers.util import cos_sim

if USE_E5_PREFIX:
    texts = [
        "query: Trí tuệ nhân tạo là gì?",
        "passage: Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "passage: Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]
else:
    texts = [
        "Trí tuệ nhân tạo là gì?",
        "Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]

embeddings = model.encode(texts)

print("Embedding shape:", embeddings.shape)
print("Relevant:", cos_sim(embeddings[0], embeddings[1]).item())
print("Unrelated:", cos_sim(embeddings[0], embeddings[2]).item())

## 7. Chọn Retrieval tasks

In [ ]:
TASK_NAMES = [
    "SciFact-VN",
    "NFCorpus-VN",
    "CQADupstackAndroid-VN",
    "SCIDOCS-VN",
    "FiQA2018-VN",
    "TRECCOVID-VN",
]

if INCLUDE_QUORA:
    TASK_NAMES.append("Quora-VN")

tasks = mteb.get_tasks(tasks=TASK_NAMES)

print("Tasks:")
for name in TASK_NAMES:
    print("-", name)

## 8. Result Cache

MTEB lưu kết quả từng task. Nếu chạy lại, `only-missing` sẽ bỏ qua phần đã có.

In [ ]:
from pathlib import Path

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

cache = mteb.ResultCache(cache_path=RESULT_DIR)

print("Cache:", RESULT_DIR)

# 9. Chạy benchmark

Task được sắp từ nhỏ đến lớn để dễ debug và có kết quả sớm.

Metric chính: **NDCG@10**.

In [ ]:
import time

total_start = time.perf_counter()

for i, task in enumerate(tasks, start=1):
    task_name = task.metadata.name

    print()
    print("=" * 70)
    print(f"[{i}/{len(tasks)}] {task_name}")
    print("=" * 70)

    start = time.perf_counter()

    mteb.evaluate(
        model,
        tasks=[task],
        cache=cache,
        encode_kwargs={"batch_size": BATCH_SIZE},
        overwrite_strategy="only-missing",
    )

    elapsed = (time.perf_counter() - start) / 60
    print(f"{task_name} finished in {elapsed:.1f} minutes")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_minutes = (time.perf_counter() - total_start) / 60
print()
print(f"Total runtime: {total_minutes:.1f} minutes")

## 10. Xem kết quả

In [ ]:
results = cache.load_results(
    tasks=tasks,
    include_remote=False,
)

df = results.to_dataframe(format="long")
display(df)

## 11. Điểm trung bình RAG Core

Đây không phải official VN-MTEB overall score.  
Đây là trung bình main score của các Retrieval task đã chọn.

In [ ]:
avg_score = df["score"].mean()
print(f"RAG Core average NDCG@10: {avg_score:.4f}")

## 12. Xuất CSV

In [ ]:
CSV_PATH = "/content/VietEmbed-RAG-V1_RAG-Core.csv"

df.to_csv(CSV_PATH, index=False)

print("Saved:", CSV_PATH)

## 13. Đóng gói cache

In [ ]:
import shutil

ZIP_PATH = "/content/VietEmbed-RAG-V1_RAG-Core-results"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    RESULT_DIR,
)

print("Saved:", ZIP_PATH + ".zip")

## 14. Download kết quả

In [ ]:
from google.colab import files

files.download("/content/VietEmbed-RAG-V1_RAG-Core.csv")
files.download("/content/VietEmbed-RAG-V1_RAG-Core-results.zip")

# Runtime dự kiến

Với E5-base, `BATCH_SIZE = 32`:

### T4 16 GB
- Không Quora: khoảng 20–60 phút
- Có Quora: khoảng 45–120 phút

### L4 24 GB
- Không Quora: khoảng 10–30 phút
- Có Quora: khoảng 20–60 phút

Nếu đánh giá V1 nghiêm túc, giữ:

```python
INCLUDE_QUORA = True
```

Nếu chỉ smoke test sau fine-tune:

```python
INCLUDE_QUORA = False
```

# Khi nào chạy full VN-MTEB Retrieval?

Chỉ nên chạy full khi model gần final hoặc chuẩn bị publish/model card, vì các task như
MSMARCO-VN, HotpotQA-VN, FEVER-VN, DBPedia-VN và NQ-VN có corpus hàng triệu documents.